In [204]:
import pandas as pd

df = pd.read_csv("output.csv")

In [205]:
from pathlib import Path
import json

explicit = []
explicit_not_perfect = []
checked = 0
for row in df.iterrows():
    section = None
    if "Research Question Likert Score" == row[1]['section']:
        section = "Research Questions"
    if "Hypothesis Likert Score" == row[1]['section']:
        section = "Hypotheses"
    if not section:
        continue
    
    data = Path("llm_output") / f"{row[1]['Paper']}.json"
    with open(data, "r") as f:
        data = json.load(f)
    
    if data[section][row[1][2]]["explicit"]:
        if row[1]['value'] == 1:
            explicit.append(row)
        else:
            explicit_not_perfect.append(row)
    

/var/folders/78/9cl0kydx0g3cdzjyk28yzr7w0000gn/T/ipykernel_58412/1475224722.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if data[section][row[1][2]]["explicit"]:


In [206]:
explicit_not_perfect

[(54,
  Paper      Edge-Based Graph Component Pooling
  section               Hypothesis Likert Score
  field                            hypothesis_1
  value                                       2
  Name: 54, dtype: object),
 (89,
  Paper      Combining Automated Optimisation of Hyperparam...
  section                              Hypothesis Likert Score
  field                                           hypothesis_2
  value                                                      2
  Name: 89, dtype: object),
 (133,
  Paper      Hyperparameters in Reinforcement Learning and ...
  section                       Research Question Likert Score
  field                                    research_question_5
  value                                                      5
  Name: 133, dtype: object),
 (230,
  Paper      Automatic Adjusting Global Similarity Measures...
  section                       Research Question Likert Score
  field                                    research_question_1
  va

In [207]:
len(explicit), len(explicit_not_perfect)

(33, 5)

# F1-Score Analysis for Element Keys

In [208]:
import json
from pathlib import Path
import pandas as pd
from collections import defaultdict

# Define paths
ground_truth_path = Path("./ground_truth")
llm_output_path = Path("./llm_output")

# --- Initialize stats dictionaries ---
# For F1-Score Analysis for Element Keys
section_stats = defaultdict(lambda: {'TP': 0, 'FP': 0, 'FN': 0})

# For F1-Score Analysis for Inter-Element Links
link_fields = [
    'research_questions', 'hypotheses', 'experiments', 
    'analyses', 'interpretations', 'conclusions'
]
link_stats = defaultdict(lambda: {'TP': 0, 'FP': 0, 'FN': 0})

# --- Define helper functions ---
def process_section_links(section_name, gt_section_data, llm_section_data):
    """Helper to calculate TP, FP, FN for inter-element links."""
    all_items = set(gt_section_data.keys()) | set(llm_section_data.keys())
    for item_key in all_items:
        gt_item = gt_section_data.get(item_key, {})
        llm_item = llm_section_data.get(item_key, {})
        for field in link_fields:
            if field in gt_item or field in llm_item:
                gt_links = set(gt_item.get(field, []))
                llm_links = set(llm_item.get(field, []))
                link_key = f"{section_name}.{field}"
                link_stats[link_key]['TP'] += len(gt_links.intersection(llm_links))
                link_stats[link_key]['FP'] += len(llm_links - gt_links)
                link_stats[link_key]['FN'] += len(gt_links - llm_links)

def calculate_and_display_metrics(stats_dict, index_name):
    """Helper to calculate and display Accuracy, Precision, Recall, and F1-Score."""
    results = []
    for key, stats in stats_dict.items():
        tp, fp, fn = stats['TP'], stats['FP'], stats['FN']
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        accuracy = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
        results.append({index_name: key, 'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1-Score': f1_score, **stats})
    
    if results:
        results_df = pd.DataFrame(results).set_index(index_name)
        results_df = results_df[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'TP', 'FP', 'FN']].sort_index()
        display(results_df)
    else:
        print(f"No data to display for {index_name} analysis.")

# --- Main processing loop ---
# Iterate over all ground truth files
for ground_truth_file in ground_truth_path.glob("*.json"):
    paper_id = ground_truth_file.stem
    llm_file = llm_output_path / f"{paper_id}.json"

    if not llm_file.exists():
        print(f"LLM output not found for {paper_id}, skipping.")
        continue

    with open(ground_truth_file) as f:
        gt_data = json.load(f)
    with open(llm_file) as f:
        llm_data = json.load(f)

    # Get all section keys from both, excluding 'Meta'
    all_sections = (set(gt_data.keys()) | set(llm_data.keys())) - {'Meta'}

    for section in all_sections:
        if section == 'Future Work':
            gt_fw_data = gt_data.get('Future Work', {})
            llm_fw_data = llm_data.get('Future Work', {})
            all_sub_sections = set(gt_fw_data.keys()) | set(llm_fw_data.keys())
            for sub_section_name in all_sub_sections:
                # Element Keys analysis for Future Work sub-sections
                gt_keys = set(gt_fw_data.get(sub_section_name, {}).keys())
                llm_keys = set(llm_fw_data.get(sub_section_name, {}).keys())
                section_stats[sub_section_name]['TP'] += len(gt_keys.intersection(llm_keys))
                section_stats[sub_section_name]['FP'] += len(llm_keys - gt_keys)
                section_stats[sub_section_name]['FN'] += len(gt_keys - llm_keys)
                
                # Inter-element links analysis for Future Work sub-sections
                gt_section_data = gt_fw_data.get(sub_section_name, {})
                llm_section_data = llm_fw_data.get(sub_section_name, {})
                process_section_links(sub_section_name, gt_section_data, llm_section_data)
        else:
            # Element Keys analysis for regular sections
            gt_keys = set(gt_data.get(section, {}).keys())
            llm_keys = set(llm_data.get(section, {}).keys())
            section_stats[section]['TP'] += len(gt_keys.intersection(llm_keys))
            section_stats[section]['FP'] += len(llm_keys - gt_keys)
            section_stats[section]['FN'] += len(gt_keys - llm_keys)
            
            # Inter-element links analysis for regular sections
            gt_section_data = gt_data.get(section, {})
            llm_section_data = llm_data.get(section, {})
            process_section_links(section, gt_section_data, llm_section_data)

# --- Display results ---
print("--- F1-Score Analysis for Element Keys ---")
calculate_and_display_metrics(section_stats, 'Section')

print("\n--- F1-Score Analysis for Inter-Element Links ---")
calculate_and_display_metrics(link_stats, 'Link Type')


--- F1-Score Analysis for Element Keys ---


,Accuracy,Precision,Recall,F1-Score,TP,FP,FN
Section,,,,,,,
Analyses,0.931818,1.000000,0.931818,0.964706,82,0,6
Conclusions,0.927273,1.000000,0.927273,0.962264,51,0,4
Experiments,1.000000,1.000000,1.000000,1.000000,73,0,0
Hypotheses,0.967742,0.967742,1.000000,0.983607,60,2,0
Interpretations,0.937500,0.989011,0.947368,0.967742,90,1,5
Research Questions,0.961039,0.973684,0.986667,0.980132,74,2,1
Suggested Hypotheses,1.000000,1.000000,1.000000,1.000000,19,0,0
Suggested Research Questions,0.892308,1.000000,0.892308,0.943089,58,0,7



--- F1-Score Analysis for Inter-Element Links ---


,Accuracy,Precision,Recall,F1-Score,TP,FP,FN
Link Type,,,,,,,
Analyses.experiments,0.934066,1.000000,0.934066,0.965909,85,0,6
Conclusions.hypotheses,0.871429,0.953125,0.910448,0.931298,61,3,6
Conclusions.interpretations,0.857143,0.943820,0.903226,0.923077,84,5,9
Conclusions.research_questions,0.873563,0.974359,0.894118,0.932515,76,2,9
Experiments.hypotheses,0.913978,0.955056,0.955056,0.955056,85,4,4
Experiments.research_questions,0.919643,0.953704,0.962617,0.958140,103,5,4
Hypotheses.research_questions,0.935897,0.948052,0.986486,0.966887,73,4,1
Interpretations.analyses,0.931373,0.989583,0.940594,0.964467,95,1,6
Suggested Hypotheses.conclusions,1.000000,1.000000,1.000000,1.000000,22,0,0


In [209]:
import json
from pathlib import Path
import pandas as pd
from collections import defaultdict

# Define paths
ground_truth_path = Path("./ground_truth")
llm_output_path = Path("./llm_output")

# Store counts for each section's properties, keyed by "Section.Property"
property_stats = defaultdict(lambda: {'TP': 0, 'FP': 0, 'FN': 0})
# New counter for total elements processed per section
element_counts = defaultdict(int)

def process_properties(section_name, gt_section_data, llm_section_data):
    """Compare properties for common elements, tracking stats per property."""
    common_item_keys = set(gt_section_data.keys()).intersection(set(llm_section_data.keys()))
    
    # Increment the count of elements processed for this section
    element_counts[section_name] += len(common_item_keys)

    for item_key in common_item_keys:
        gt_item = gt_section_data.get(item_key, {})
        llm_item = llm_section_data.get(item_key, {})

        # Exclude 'note' and 'reason' from comparison
        gt_props = {k for k in gt_item.keys() if k not in ['note', 'reason']}
        llm_props = {k for k in llm_item.keys() if k not in ['note', 'reason']}

        # Properties in both: check for match (TP) or mismatch (FP+FN)
        for prop_key in gt_props.intersection(llm_props):
            stats_key = f"{section_name}.{prop_key}"
            gt_value = gt_item.get(prop_key)
            llm_value = llm_item.get(prop_key)
            
            # Sanity check counter
            initial_sum = sum(property_stats[stats_key].values())

            if prop_key == 'results' and section_name == 'Analyses':
                total_tp, total_fp, total_fn = 0, 0, 0
                
                result_sub_keys = set(gt_value.keys()) | set(llm_value.keys())
                for sub_key in result_sub_keys: # Figures, Tables, Text
                    sub_stats_key = f"{stats_key}.{sub_key}"
                    gt_sub_dict = gt_value.get(sub_key, {})
                    llm_sub_dict = llm_value.get(sub_key, {})
                    
                    gt_sub_keys = set(gt_sub_dict.keys())
                    llm_sub_keys = set(llm_sub_dict.keys())
                    
                    tp = 0
                    fn_set = set()
                    fp_set = set()

                    if sub_key == 'Text':
                        # Stricter check for Text: keys and their 'value' property must match
                        common_text_keys = gt_sub_keys.intersection(llm_sub_keys)
                        for text_key in common_text_keys:
                            gt_text_val = gt_sub_dict.get(text_key, {}).get('value')
                            llm_text_val = llm_sub_dict.get(text_key, {}).get('value')
                            if str(gt_text_val) == str(llm_text_val):
                                tp += 1
                            else:
                                # Mismatch in value for a common key
                                print(f"{section_name}\t{prop_key}.{sub_key}.{text_key}\t\tMismatch: GT='{gt_text_val}', LLM='{llm_text_val}'")
                                fn_set.add(text_key)
                                fp_set.add(text_key)
                        
                        # Keys only in GT (FN) or LLM (FP)
                        fn_set.update(gt_sub_keys - llm_sub_keys)
                        fp_set.update(llm_sub_keys - gt_sub_keys)
                    else: # For Figures and Tables, just compare keys
                        tp = len(gt_sub_keys.intersection(llm_sub_keys))
                        fn_set = gt_sub_keys - llm_sub_keys
                        fp_set = llm_sub_keys - gt_sub_keys
                    
                    if fn_set:
                        print(f"{section_name}\t{prop_key}.{sub_key}\t\tFN: {fn_set}")
                    if fp_set:
                        print(f"{section_name}\t{prop_key}.{sub_key}\t\tFP: {fp_set}")

                    property_stats[sub_stats_key]['TP'] += tp
                    property_stats[sub_stats_key]['FN'] += len(fn_set)
                    property_stats[sub_stats_key]['FP'] += len(fp_set)
                    
                    total_tp += tp
                    total_fn += len(fn_set)
                    total_fp += len(fp_set)
                
                # Add combined total for the main 'results' property
                property_stats[stats_key]['TP'] += total_tp
                property_stats[stats_key]['FN'] += total_fn
                property_stats[stats_key]['FP'] += total_fp

            elif isinstance(gt_value, dict) and isinstance(llm_value, dict):
                gt_keys = set(gt_value.keys())
                llm_keys = set(llm_value.keys())
                
                fn_set = gt_keys - llm_keys
                fp_set = llm_keys - gt_keys
                if fn_set:
                    print(f"{section_name}\t{prop_key}\t\tFN: {fn_set}")
                if fp_set:
                    print(f"{section_name}\t{prop_key}\t\tFP: {fp_set}")

                property_stats[stats_key]['TP'] += len(gt_keys.intersection(llm_keys))
                property_stats[stats_key]['FN'] += len(fn_set)
                property_stats[stats_key]['FP'] += len(fp_set)
                # If both are empty dicts, it's a match.
                if not gt_keys and not llm_keys:
                    property_stats[stats_key]['TP'] += 1
            elif isinstance(gt_value, list) and isinstance(llm_value, list):
                gt_set = set(map(str, gt_value))
                llm_set = set(map(str, llm_value))

                fn_set = gt_set - llm_set
                fp_set = llm_set - gt_set
                if fn_set:
                    print(f"{section_name}\t{prop_key}\t\tFN: {fn_set}")
                if fp_set:
                    print(f"{section_name}\t{prop_key}\t\tFP: {fp_set}")

                property_stats[stats_key]['TP'] += len(gt_set.intersection(llm_set))
                property_stats[stats_key]['FN'] += len(fn_set)
                property_stats[stats_key]['FP'] += len(fp_set)
                # If both are empty lists, it's a match.
                if not gt_set and not llm_set:
                    property_stats[stats_key]['TP'] += 1
            else:
                if str(gt_value) == str(llm_value):
                    property_stats[stats_key]['TP'] += 1
                else:
                    # A single mismatch is one FN (for the ground truth) and one FP (for the LLM's output)
                    print(f"{section_name}\t{prop_key}\t\tMismatch: GT='{gt_value}', LLM='{llm_value}'")
                    property_stats[stats_key]['FN'] += 1
                    property_stats[stats_key]['FP'] += 1
            
            # Assert that one of the counters was incremented
            assert sum(property_stats[stats_key].values()) > initial_sum, f"Property '{stats_key}' was not counted."

        # Properties in GT, but not in LLM (False Negative)
        for prop_key in gt_props - llm_props:
            stats_key = f"{section_name}.{prop_key}"
            gt_value = gt_item.get(prop_key)
            print(f"{section_name}\t{prop_key}\t\tFN: Missing in LLM (GT value: '{gt_value}')")
            if isinstance(gt_value, dict):
                property_stats[stats_key]['FN'] += len(gt_value.keys())
            elif isinstance(gt_value, list):
                property_stats[stats_key]['FN'] += len(gt_value)
            else:
                property_stats[stats_key]['FN'] += 1

        # Properties in LLM, but not in GT (False Positive)
        for prop_key in llm_props - gt_props:
            stats_key = f"{section_name}.{prop_key}"
            llm_value = llm_item.get(prop_key)
            print(f"{section_name}\t{prop_key}\t\tFP: Extra in LLM (LLM value: '{llm_value}')")
            if isinstance(llm_value, dict):
                property_stats[stats_key]['FP'] += len(llm_value.keys())
            elif isinstance(llm_value, list):
                property_stats[stats_key]['FP'] += len(llm_value)
            else:
                property_stats[stats_key]['FP'] += 1

# --- Main processing loop ---
for ground_truth_file in ground_truth_path.glob("*.json"):
    paper_id = ground_truth_file.stem
    llm_file = llm_output_path / f"{paper_id}.json"

    if not llm_file.exists():
        continue

    with open(ground_truth_file) as f:
        gt_data = json.load(f)
    with open(llm_file) as f:
        llm_data = json.load(f)

    all_sections = (set(gt_data.keys()) | set(llm_data.keys())) - {'Meta'}

    for section_name in all_sections:
        if section_name == 'Future Work':
            gt_fw_data = gt_data.get('Future Work', {})
            llm_fw_data = llm_data.get('Future Work', {})
            all_sub_sections = set(gt_fw_data.keys()) | set(llm_fw_data.keys())
            for sub_section_name in all_sub_sections:
                gt_section_data = gt_fw_data.get(sub_section_name, {})
                llm_section_data = llm_fw_data.get(sub_section_name, {})
                process_properties(sub_section_name, gt_section_data, llm_section_data)
        else:
            gt_section_data = gt_data.get(section_name, {})
            llm_section_data = llm_data.get(section_name, {})
            process_properties(section_name, gt_section_data, llm_section_data)

# --- Display results with Multi-level Index ---
print("\n--- Total elements processed per section ---")
for section, count in sorted(element_counts.items()):
    print(f"{section}: {count}")
print("\n")

print("--- F1-Score Analysis for Element Properties ---")

# Create a list of dictionaries for the DataFrame
prop_results = []
for key, stats in property_stats.items():
    tp, fp, fn = stats['TP'], stats['FP'], stats['FN']
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
    
    # Gracefully handle keys that might not have a '.'
    if '.' in key:
        parts = key.split('.', 2)
        if len(parts) > 2:
             section, prop = parts[0], '.'.join(parts[1:])
        else:
            section, prop = parts
    else:
        section, prop = key, ''
        
    prop_results.append({
        'Section': section,
        'Property': prop,
        'Accuracy': accuracy, 
        'Precision': precision, 
        'Recall': recall, 
        'F1-Score': f1_score, 
        **stats
    })

if prop_results:
    # Create DataFrame and set the multi-level index
    prop_results_df = pd.DataFrame(prop_results).set_index(['Section', 'Property'])
    
    # Reorder columns and sort by the index
    prop_results_df = prop_results_df[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'TP', 'FP', 'FN']].sort_index()
    
    display(prop_results_df)
else:
    print("No data to display for Element Properties analysis.")


Suggested Hypotheses	value		Mismatch: GT='The authors hypothesise that using top-k pool or edge pool selection strategies instead of a threshold might better align with user objectives regarding the number of merged edges and that including this into the method may serve a wider range of users.', LLM='The authors hypothesise that using top-k pool or edge pool selection strategies instead of a threshold might better align with user objectives regarding the number of merged edges.'
Suggested Hypotheses	value		Mismatch: GT='The authors hypothesise that including edge features in the scoring method would allow edge features to have a direct impact on the features of the newly created node and thus make the method applicable to graphs with edge features.', LLM='The authors hypothesise that including edge features in the scoring method would allow edge features to have a direct impact on the features of the newly created node.'
Experiments	strategy		Mismatch: GT='Random split (train = 0.8, v

Accuracy  Precision  \
Section                      Property                                      
Analyses                     experiments             1.000000   1.000000   
                             metrics                 0.976744   0.976744   
                             results                 0.879859   0.980315   
                             results.Figures         0.794118   0.981818   
                             results.Tables          0.973684   0.973684   
                             results.Text            0.943662   0.985294   
                             statistics              0.929032   0.941176   
                             test                    0.956989   0.978022   
Conclusions                  hypotheses              0.957143   0.957143   
                             interpretations         0.913043   0.943820   
                             research_questions      0.962500   0.974684   
                             support                 0.942308   0.960784   
                             value                   0.645161   0.784314   
Experiments                  data                    0.967593   0.981221   
                             experiment_description  0.921053   0.958904   
                             hypotheses              0.918367   0.957447   
                             research_questions      0.919643   0.953704   
                             strategy                0.622222   0.767123   
Hypotheses                   explicit                1.000000   1.000000   
                             research_questions      0.960526   0.973333   
                             value                   0.578947   0.733333   
Interpretations              analyses                0.989583   1.000000   
                             value                   0.818182   0.900000   
Research Questions           explicit                0.973333   0.986486   
                             value                   0.741176   0.851351   
Suggested Hypotheses         conclusions             1.000000   1.000000   
                             value                   0.583333   0.736842   
Suggested Research Questions conclusions             1.000000   1.000000   
                             value                   0.901639   0.948276   

                                                       Recall  F1-Score   TP  \
Section                      Property                                          
Analyses                     experiments             1.000000  1.000000   85   
                             metrics                 1.000000  0.988235  168   
                             results                 0.895683  0.936090  249   
                             results.Figures         0.805970  0.885246  108   
                             results.Tables          1.000000  0.986667   74   
                             results.Text            0.957143  0.971014   67   
                             statistics              0.986301  0.963211  144   
                             test                    0.978022  0.978022   89   
Conclusions                  hypotheses              1.000000  0.978102   67   
                             interpretations         0.965517  0.954545   84   
                             research_questions      0.987179  0.980892   77   
                             support                 0.980000  0.970297   49   
                             value                   0.784314  0.784314   40   
Experiments                  data                    0.985849  0.983529  209   
                             experiment_description  0.958904  0.958904   70   
                             hypotheses              0.957447  0.957447   90   
                             research_questions      0.962617  0.958140  103   
                             strategy                0.767123  0.767123   56   
Hypotheses                   explicit                1.000000  1.000000   60   
                